# Classification of Consumer Complaints

The Consumer Financial Protection Bureau publishes the Consumer Complaint Database, a collection of complaints about consumer financial products and services that were sent to companies for response. Complaints are published after the company responds, confirming a commercial relationship with the consumer, or after 15 days, whichever comes first. 

You have been provided with a dataset of over 350,000 such complaints for 5 common issue types. Your goal is to train a text classification model to identify the issue type based on the consumer complaint narrative. The data can be downloaded from https://drive.google.com/file/d/1Hz1gnCCr-SDGjnKgcPbg7Nd3NztOLdxw/view?usp=share_link 

As you work, answer the following questions: 
* What steps did you take to preprocess the data?
* How did a model using unigrams compare to one using bigrams or trigrams?
* How did a count vectorizer compare to a tfidf vectorizer?
* What models did you try and how successful were they? Where did they struggle? Were there issues that the models commonly mixed up?
* What words or phrases were most influential on your models' predictions?

**Bonus:** A larger dataset containing 20 additional categories can be downloaded from https://drive.google.com/file/d/1gW6LScUL-Z7mH6gUZn-1aNzm4p4CvtpL/view?usp=share_link. How well do your models work with these additional categories?

In [1]:
import pandas as pd
import numpy as np

from joblib import dump, load

from sklearn.naive_bayes import MultinomialNB

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
complaints = pd.read_csv("../data/complaints.csv")

In [3]:
complaints.head()

,Consumer complaint narrative,Issue
0,My name is XXXX XXXX this complaint is not mad...,Incorrect information on your report
1,I searched on XXXX for XXXXXXXX XXXX and was ...,Fraud or scam
2,I have a particular account that is stating th...,Incorrect information on your report
3,I have not supplied proof under the doctrine o...,Attempts to collect debt not owed
4,Hello i'm writing regarding account on my cred...,Incorrect information on your report


In [4]:
complaints['Issue'].value_counts().sort_index()

Issue
Attempts to collect debt not owed        73163
Communication tactics                    21243
Fraud or scam                            12347
Incorrect information on your report    229305
Struggling to pay mortgage               17374
Name: count, dtype: int64

In [5]:
seed = 123
for statement in complaints.loc[complaints['Issue'] == 'Attempts to collect debt not owed', 'Consumer complaint narrative'].sample(3, random_state=seed):
    print(statement)
    print('-----------------------------')

This company in which I hold no contract with nor have received services from reported ( 3 ) different collection accounts against my Social Security Number in the amount of {$510.00}, {$710.00} and {$570.00} with XXXX, XXXX & XXXX   credit reporting agencies. I requested verification and validation on XX/XX/2018 of the alleged debt and account, however, the business failed to provide adequate proof. Considering this business does not have a contract with me for goods or services they have provided nor have they provided adequate proof, I am not obligated to pay for the alleged debt.
-----------------------------
An affidavit of Billing Error Notice was mailed to XXXX XXXX XXXX XXXX and/or XXXX, XXXX but they didn't respond back. 

The account is an agreement, not a contract. Based on the consumer protection laws and your lack of complete disclosure I rescind the entire transaction due to fraud. 

CONSUMER PROTECTION LAWS and U.S. CODE VIOLATIONS : Equal Credit Opportunity Act / Truth 

In [6]:
seed = 123
for statement in complaints.loc[complaints['Issue'] == 'Communication tactics', 'Consumer complaint narrative'].sample(3, random_state=seed):
    print(statement)
    print('-----------------------------')

The company name is Valentine and Kebartas. 
After missimg multiple calls a day from this company I finally spoke with someone on XXXX/XXXX/16. XXXX had sent my final bill to my old address and I never got it. The person I spoke to at Valentine and Kebartas corrected my address and arranged to send out a reprint of the bill. She waved the ridiculous {$5.00} fee to have the bill reprinted. I let her know that I would be taking care of the bill as soon as I received it. 
Not 1 day later the calls started again. 
I received a call this morning by a very pushy caller and was told that if I was taken off the call list without making payment arrangements my bill would go into collections. I asked why my file had n't been updated to show that I was cooperating and s ( he ) said their system just does n't show everything. 
When I complained about their repetitive calls the caller said that legally the system could call my phone up to 6 times per day. This is harrassment and also threatening by

In [7]:
X = complaints[['Consumer complaint narrative']]
y = complaints['Issue']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 321, stratify = y)

In [8]:
vect = CountVectorizer()

X_train_vec = vect.fit_transform(X_train['Consumer complaint narrative'])
X_test_vec = vect.transform(X_test['Consumer complaint narrative'])

In [9]:
X_train_vec

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 21821225 stored elements and shape (265074, 72222)>

In [10]:
nb = MultinomialNB().fit(X_train_vec, y_train)

y_pred = nb.predict(X_test_vec)

In [11]:
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7988976663120487
[[12086  2039   500  3343   323]
 [  587  4476    51   112    85]
 [   67    55  2813   110    42]
 [ 6610  1046   834 46997  1839]
 [   36    40    12    38  4217]]


In [12]:
word = 'great'

probs = np.exp(nb.feature_log_prob_)[:, vect.vocabulary_[word]]

In [13]:
for label, prob in zip(nb.classes_, probs):
    print(f"Class: {label:<20} | Probability: {prob:.8f}")

Class: Attempts to collect debt not owed | Probability: 0.00004856
Class: Communication tactics | Probability: 0.00004795
Class: Fraud or scam        | Probability: 0.00006866
Class: Incorrect information on your report | Probability: 0.00005552
Class: Struggling to pay mortgage | Probability: 0.00007906


In [14]:
vect = CountVectorizer()
clf = MultinomialNB()

pipe = Pipeline([("vect", vect), ("clf", clf)])

param_grid = {
    'vect__ngram_range':[(1,1), (1,2), (1,3)],
    'vect__min_df':[1, 2, 5, 10, 20],
    'clf__fit_prior':[False, True]
}

In [ ]:
rs = RandomizedSearchCV(estimator = pipe, param_distributions = param_grid, verbose = 2, n_jobs = -1)
rs.fit(X_train['Consumer complaint narrative'], y_train)

dump(rs, "../models/cv_01.joblib")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END clf__fit_prior=True, vect__min_df=2, vect__ngram_range=(1, 1); total time=  18.1s
[CV] END clf__fit_prior=False, vect__min_df=5, vect__ngram_range=(1, 2); total time= 1.0min
[CV] END clf__fit_prior=False, vect__min_df=10, vect__ngram_range=(1, 2); total time= 1.7min
[CV] END clf__fit_prior=False, vect__min_df=10, vect__ngram_range=(1, 2); total time= 1.6min


Exception ignored in: <function ResourceTracker.__del__ at 0x106ecdbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END clf__fit_prior=False, vect__min_df=5, vect__ngram_range=(1, 2); total time=  59.3s
[CV] END clf__fit_prior=True, vect__min_df=5, vect__ngram_range=(1, 3); total time= 4.1min


Exception ignored in: <function ResourceTracker.__del__ at 0x107019bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


In [ ]:
* What steps did you take to preprocess the data?
* How did a model using unigrams compare to one using bigrams or trigrams?
* How did a count vectorizer compare to a tfidf vectorizer?
* What models did you try and how successful were they? Where did they struggle? Were there issues that the models commonly mixed up?
* What words or phrases were most influential on your models' predictions?